# Dataset A: llimllib/nba_data, Team Records, Playoff Results, and 5-Year Playoff History (2015-16 to 2025-26)

This notebook builds most of the **Team Stats** features from `ListOfPotentialFeatures.md`:

| Feature | Column(s) | Level |
|---|---|---|
| Games won | `wins` (also `win_pct`) | team-season |
| Made the playoffs (yes/no) | `made_playoffs` | team-season |
| Furthest playoff round reached | `furthest_playoff_round` (0 = missed … 4 = lost in the Finals, 5 = champion) | team-season |
| 5-year trend: playoff appearances | `playoff_appearances_last_5` | team-season |
| 5-year trend: playoff round reached | `avg_playoff_round_last_5` | team-season |

The output is **one row per team + season**. The remaining Team Stats feature, the **regular-season final standing (seed)**, comes from ESPN's official standings in notebook `02`. Notebook `999` combines the two into one team table. Every player row for that team and season gets these values when that table is joined onto the player tables on `season` + `team`.

*Run order: `01` → `02` → `999`. Notebook 02 also reads two of the raw files this notebook downloads.*

**Why the current season's results are usable:** we only predict **offseason** trades. Each season's standings and playoff results are final before the offseason that follows, so none of these columns use information from after the trade window opens.

- **Dataset name:** llimllib/nba_data, a public DuckDB/parquet catalog of NBA data rebuilt every 4 hours (the same source as `salary-data/02`)
- **Link:** https://github.com/llimllib/nba_data (data is served from `https://basketball-data.billmill.org/nba/nba.duckdb`)
- **Tables used:**
  - `team_seasons`: NBA team id, abbreviation, full name, and conference for each season.
  - `bbref_team_stats`: Basketball-Reference team totals. We use `made_playoffs`, `wins`, `losses`, and `g` (games played).
  - `games`: one row per game, with `game_type` (`playoffs`, `play_in`, ...) and `playoff_round` (1-4).
  - `team_game_logs`: one row per team per game, with a `win` flag. We use it only to find who won each Finals.
- **Saved as:** `data-files/00-raw/llimllib_nba_team_seasons_raw.csv`, `llimllib_bbref_team_stats_raw.csv`, `llimllib_playoff_games_raw.csv`, `llimllib_playoff_team_game_logs_raw.csv` (downloaded below if missing)

## Important variables

| Variable | Units / values | Meaning |
|---|---|---|
| `season` | year | The year the season **ends** (2026 = 2025-26). Same convention as our other notebooks. |
| `team` | text | NBA 3-letter abbreviation (`GSW`, `NYK`, ...). The join key with the other tables. |
| `team_id` | text | NBA team id (e.g. `1610612744` = Golden State). Stays with the franchise through renames and moves. |
| `full_name` | text | Full team name. |
| `conference` | East / West | The team's conference that season. |
| `wins` | games | Regular-season wins. |
| `win_pct` | 0-1 | `wins / g`. Use this to compare seasons, since 2019-20 and 2020-21 had fewer than 82 games. |
| `made_playoffs` | True / False | Qualified for the playoffs. Teams that lost in the play-in count as **not** making the playoffs. |
| `furthest_playoff_round` | 0-5 | 0 = missed the playoffs, 1 = lost in round 1, 2 = lost in the conference semifinals, 3 = lost in the conference finals, 4 = lost in the Finals, 5 = won the championship. |
| `playoff_appearances_last_5` | 0-5 | Playoff appearances in the **five seasons before** this one (the current season is not counted). |
| `avg_playoff_round_last_5` | 0-5 | Mean of `furthest_playoff_round` over the five seasons before this one, counting missed seasons as 0. |

## Concerns going in

1. **The standings are not computed here.** Ranking teams by win % would give teams with identical records the same rank, and the official order between them is set by the NBA's tiebreakers (head-to-head record, division and conference record, and so on). Those ties are common, and they often fall right at the playoff and play-in cutoffs. So the seed comes from ESPN's official standings in notebook 02 rather than being approximated here.
2. **The 5-year trends need earlier seasons.** Like the 3-year player averages in `player-stats`, the 5-year team history for 2015-16 needs the five seasons before it. So we also load **2010-11 through 2014-15**. Those seasons are used only to compute the trends and are not rows in the output.
3. **These are team-level features.** Team success may affect roster decisions, but contracts, age, injuries, and individual performance matter too. That is why this table is combined with the player and salary tables.
4. **Past results don't guarantee future results.** Injuries, coaching changes, and roster moves can change a team quickly. The 5-year trends add context but can't capture those changes.

## Setup

In [1]:
import sys
from pathlib import Path

import duckdb
import pandas as pd

# The shared helpers (paths, season range) live next to the salary notebooks.
sys.path.insert(0, str(Path('..') / 'salary-data'))
from nba_cleaning_utils import RAW, PROCESSED, FIRST_SEASON, LAST_SEASON

# The 5-year trends for FIRST_SEASON need the five seasons before it.
HISTORY_FIRST = FIRST_SEASON - 5
print('output seasons:', FIRST_SEASON, 'to', LAST_SEASON, '| history loaded from:', HISTORY_FIRST)

output seasons: 2016 to 2026 | history loaded from: 2011


## 1. Download (skipped if the raw files already exist)

DuckDB reads the remote catalog over HTTPS and downloads only the parquet files a query needs. We save the raw tables as CSV so the rest of the notebook runs offline. We download every season and filter in pandas below.

In [2]:
RAW_TEAMS = RAW / 'llimllib_nba_team_seasons_raw.csv'
RAW_STATS = RAW / 'llimllib_bbref_team_stats_raw.csv'
RAW_GAMES = RAW / 'llimllib_playoff_games_raw.csv'
RAW_LOGS = RAW / 'llimllib_playoff_team_game_logs_raw.csv'

if not all(p.exists() for p in (RAW_TEAMS, RAW_STATS, RAW_GAMES, RAW_LOGS)):
    con = duckdb.connect()
    con.sql('INSTALL httpfs; LOAD httpfs;')
    con.sql("ATTACH 'https://basketball-data.billmill.org/nba/nba.duckdb' AS nba (READ_ONLY)")
    con.sql('SELECT * FROM nba.team_seasons WHERE is_nba ORDER BY season, team_id').df().to_csv(RAW_TEAMS, index=False)
    con.sql('''SELECT season, team_id, made_playoffs, wins, losses, g
               FROM nba.bbref_team_stats ORDER BY season, team_id''').df().to_csv(RAW_STATS, index=False)
    con.sql('''SELECT season, game_id, game_type, playoff_round, series_number, series_game, home_team_id, away_team_id
               FROM nba.games WHERE game_type IN ('playoffs', 'play_in')
               ORDER BY season, game_id''').df().to_csv(RAW_GAMES, index=False)
    con.sql('''SELECT season, game_id, team_id, win
               FROM nba.team_game_logs
               WHERE game_id IN (SELECT game_id FROM nba.games WHERE game_type = 'playoffs')
               ORDER BY season, game_id, team_id''').df().to_csv(RAW_LOGS, index=False)
    print('downloaded the four tables')
else:
    print('raw files already present, skipping download')

raw files already present, skipping download


## 2. Load the data

In [3]:
ids = {'team_id': str, 'home_team_id': str, 'away_team_id': str, 'game_id': str}
teams_raw = pd.read_csv(RAW_TEAMS, dtype=ids)
stats_raw = pd.read_csv(RAW_STATS, dtype=ids)
games_raw = pd.read_csv(RAW_GAMES, dtype=ids)
logs_raw = pd.read_csv(RAW_LOGS, dtype=ids)

teams_raw.head()

,season,team_id,nba_abbrev,full_name,is_nba,conference,division,cup_group
0,1997,1610612737,ATL,Atlanta Hawks,True,East,Central,NaN
1,1997,1610612738,BOS,Boston Celtics,True,East,Atlantic,NaN
2,1997,1610612739,CLE,Cleveland Cavaliers,True,East,Central,NaN
3,1997,1610612741,CHI,Chicago Bulls,True,East,Central,NaN
4,1997,1610612742,DAL,Dallas Mavericks,True,West,Midwest,NaN


In [4]:
stats_raw.head(3)

,season,team_id,made_playoffs,wins,losses,g
0,2010,1610612737,True,53,29,82
1,2010,1610612738,True,50,32,82
2,2010,1610612739,True,61,21,82


In [5]:
games_raw.head(3)

,season,game_id,game_type,playoff_round,series_number,series_game,home_team_id,away_team_id
0,1997,0049600001,playoffs,0,0,1,1610612752,1610612766
1,1997,0049600002,playoffs,0,0,2,1610612748,1610612753
2,1997,0049600003,playoffs,0,0,3,1610612745,1610612750


## 3A. Is the data tidy?

Tidy means one row per observation and one column per variable. For `team_seasons` and `bbref_team_stats` the observation should be a team-season. For `games` it is one game.

In [6]:
for name, df in [('team_seasons', teams_raw), ('bbref_team_stats', stats_raw)]:
    print(f'{name}: duplicate (season, team_id) rows =', df.duplicated(['season', 'team_id']).sum())
print('games: duplicate game_id rows =', games_raw['game_id'].duplicated().sum())
print('team_game_logs: duplicate (game_id, team_id) rows =', logs_raw.duplicated(['game_id', 'team_id']).sum())

team_seasons: duplicate (season, team_id) rows = 0
bbref_team_stats: duplicate (season, team_id) rows = 0
games: duplicate game_id rows = 0
team_game_logs: duplicate (game_id, team_id) rows = 0


In [7]:
in_range = teams_raw.query('@HISTORY_FIRST <= season <= @LAST_SEASON')
in_range.groupby('season')['team_id'].nunique().rename('NBA teams').to_frame().T

season,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
NBA teams,30,30,30,30,30,30,30,30,30,30,30,30,30,30,30,30


**What we found:** `team_seasons` and `bbref_team_stats` are already tidy, with one row per team-season, no duplicate keys, and 30 NBA teams in every season we use. Both share `season` + `team_id`, so they merge directly.

`games` is tidy for what it records (one row per game), but each game has **two** teams, a home team and an away team, in separate columns. To get "the furthest round each team reached," we reshape it in section 4A into one row per team per game.

## 3B. Size of the data

In [8]:
for name, df in [('team_seasons', teams_raw), ('bbref_team_stats', stats_raw),
                 ('playoff + play-in games', games_raw), ('playoff team game logs', logs_raw)]:
    print(f'{name:24s} rows, columns: {df.shape} | seasons {df["season"].min()}-{df["season"].max()}')

team_seasons             rows, columns: (922, 8) | seasons 1997-2027
bbref_team_stats         rows, columns: (540, 6) | seasons 2010-2027
playoff + play-in games  rows, columns: (2477, 8) | seasons 1997-2026
playoff team game logs   rows, columns: (4880, 4) | seasons 1997-2026


In [9]:
team_seasons = teams_raw.query('@FIRST_SEASON <= season <= @LAST_SEASON').copy()
team_stats = stats_raw.query('@FIRST_SEASON <= season <= @LAST_SEASON').copy()
print('team-seasons in our 2015-16 to 2025-26 window:', len(team_seasons), '(team_seasons) /', len(team_stats), '(bbref_team_stats)')

team-seasons in our 2015-16 to 2025-26 window: 330 (team_seasons) / 330 (bbref_team_stats)


The source goes back to the late 1990s and runs through the current 2026-27 season, which is still in progress, so it has no final standings or playoffs yet. We keep our 11 seasons, **2016-2026 in end years**: 30 teams × 11 seasons = **330 team-seasons**. The partial 2026-27 season is dropped. The five seasons from 2011 to 2015 are kept separately, for the 5-year trends only.

## 3C. Missing data

In [10]:
team_seasons.isna().sum()

season          0
team_id         0
nba_abbrev      0
full_name       0
is_nba          0
conference      0
division        0
cup_group     240
dtype: int64

In [11]:
team_stats.isna().sum()

season           0
team_id          0
made_playoffs    0
wins             0
losses           0
g                0
dtype: int64

In [12]:
games_raw.query('@HISTORY_FIRST <= season <= @LAST_SEASON').isna().sum()

season           0
game_id          0
game_type        0
playoff_round    0
series_number    0
series_game      0
home_team_id     0
away_team_id     0
dtype: int64

In [13]:
team_seasons.groupby('season')['cup_group'].apply(lambda s: s.isna().sum()).rename('cup_group blank').to_frame().T

season,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
cup_group blank,30,30,30,30,30,30,30,30,0,0,0


**What we found:** The only blanks are in `cup_group`, which is blank for all 30 teams through 2022-23 (end year 2023) and filled in from 2023-24 on. **That is not missing data. The NBA Cup (in-season tournament) started in 2023-24.** `cup_group` isn't one of our features, so we drop the column rather than fill it. No rows are lost.

`bbref_team_stats` and the playoff/play-in `games` have no blanks at all in 2011-2026.

## 3D. Outliers and suspicious entries

Extreme seasons (a 73-win team, a 10-win team) are real NBA outcomes, so we check for **impossible** values instead: negative wins, wins + losses that don't add up to games played, playoff rounds outside 1-4, a team listed as playing itself, and playoff brackets that don't have 16 teams and 2 finalists.

In [14]:
team_stats[['wins', 'losses', 'g']].describe().round(1)

,wins,losses,g
count,330.0,330.0,330.0
mean,40.0,40.0,80.1
std,12.2,12.0,4.3
min,10.0,9.0,64.0
25%,31.2,31.2,82.0
50%,42.0,39.0,82.0
75%,49.0,48.8,82.0
max,73.0,72.0,82.0


In [15]:
print('records where wins/losses are negative or wins + losses != games:',
      ((team_stats['wins'] < 0) | (team_stats['losses'] < 0)
       | (team_stats['wins'] + team_stats['losses'] != team_stats['g'])).sum())
print('\ngames played by season (min-max):')
print(team_stats.groupby('season')['g'].agg(['min', 'max']).T)

records where wins/losses are negative or wins + losses != games: 0

games played by season (min-max):
season  2016  2017  2018  2019  2020  2021  2022  2023  2024  2025  2026
min       82    82    82    82    64    72    82    82    82    82    82
max       82    82    82    82    75    72    82    82    82    82    82


In [16]:
games = games_raw.query('@HISTORY_FIRST <= season <= @LAST_SEASON').copy()
print('game types and rounds:')
print(games.groupby(['game_type', 'playoff_round']).size().rename('games').to_string())
print('\ngames where a team plays itself:', (games['home_team_id'] == games['away_team_id']).sum())

game types and rounds:
game_type  playoff_round
play_in    1                 25
           2                 12
playoffs   1                702
           2                368
           3                178
           4                 91

games where a team plays itself: 0


In [17]:
po = games.query("game_type == 'playoffs'")
bracket = pd.DataFrame({
    'playoff teams': po.groupby('season').apply(lambda d: pd.concat([d['home_team_id'], d['away_team_id']]).nunique()),
    'Finals teams': po.query('playoff_round == 4').groupby('season').apply(lambda d: pd.concat([d['home_team_id'], d['away_team_id']]).nunique()),
})
bracket.T

season,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
playoff teams,16,16,16,16,16,16,16,16,16,16,16,16,16,16,16,16
Finals teams,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2


**What we found:**

- **Wins run from 10 to 73 and every record adds up** (wins + losses = games). Teams played fewer than 82 games in two seasons, both real: 2019-20 (64-75 games, because of the COVID-19 shutdown and the bubble) and 2020-21 (72 games for every team). This is why we also keep `win_pct`.
- **Play-in games are a separate `game_type`** (`play_in`, from 2020-21 on). We count only `game_type == 'playoffs'` toward playoff rounds, so a team that lost in the play-in correctly counts as missing the playoffs.
- **Every season, 2011 through 2026, has exactly 16 playoff teams and 2 Finals teams,** playoff rounds are all 1-4, and no team plays itself.

No outliers or impossible entries were found, so every row is kept.

## 4. Clean the data and build the features

### 4A. Furthest playoff round, with the champion as round 5

We stack the home and away teams so each playoff game gives one row per team, then take each team's highest round. That gives 1-4. **Both Finals teams reach round 4**, so to separate the champion from the runner-up we count Finals wins in `team_game_logs`: the team with 4 Finals wins is the champion and gets **5**.

In [18]:
po_long = pd.concat([
    po[['season', 'playoff_round', 'home_team_id']].rename(columns={'home_team_id': 'team_id'}),
    po[['season', 'playoff_round', 'away_team_id']].rename(columns={'away_team_id': 'team_id'}),
], ignore_index=True)

rounds = (po_long.groupby(['season', 'team_id'], as_index=False)['playoff_round'].max()
                 .rename(columns={'playoff_round': 'furthest_playoff_round'}))

# Finals wins per team (Finals = round 4)
finals_ids = po.loc[po['playoff_round'] == 4, 'game_id']
finals_wins = (logs_raw[logs_raw['game_id'].isin(finals_ids)]
               .groupby(['season', 'team_id'], as_index=False)['win'].sum()
               .rename(columns={'win': 'finals_wins'}))
champions = finals_wins.loc[finals_wins['finals_wins'] == 4, ['season', 'team_id']]
assert champions['season'].is_unique and len(champions) == po['season'].nunique(), 'expected one champion per season'

is_champ = rounds.set_index(['season', 'team_id']).index.isin(champions.set_index(['season', 'team_id']).index)
rounds.loc[is_champ, 'furthest_playoff_round'] = 5

# Show the champion and runner-up for each season
names = teams_raw[['season', 'team_id', 'nba_abbrev']]
finals_table = (finals_wins.merge(names, on=['season', 'team_id'])
                .sort_values(['season', 'finals_wins'], ascending=[True, False])
                .groupby('season')
                .agg(champion=('nba_abbrev', 'first'), runner_up=('nba_abbrev', 'last'),
                     series=('finals_wins', lambda w: f'{w.iloc[0]}-{w.iloc[1]}')))
finals_table.T

season,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
champion,DAL,MIA,MIA,SAS,GSW,CLE,GSW,GSW,TOR,LAL,MIL,GSW,DEN,BOS,OKC,NYK
runner_up,MIA,OKC,SAS,MIA,CLE,GSW,CLE,CLE,GSW,MIA,PHX,BOS,MIA,DAL,IND,SAS
series,4-2,4-1,4-3,4-1,4-2,4-3,4-1,4-0,4-2,4-2,4-2,4-2,4-1,4-1,4-3,4-1


**What we found:** Every season has exactly one team with 4 Finals wins, and the champions match the real ones (for example, CLE over GSW in 2016 after trailing the series 3-1, TOR in 2019, the LAL bubble title in 2020, OKC in 2025, and NYK in 2026).

### 4B. Join the rounds to playoff qualification and check that they agree

`bbref_team_stats.made_playoffs` and our playoff-games table are two independent sources for the same fact. A team should have a playoff round **if and only if** `made_playoffs` is True. After checking that they agree, teams that missed the playoffs get round 0.

In [19]:
history = (stats_raw.query('@HISTORY_FIRST <= season <= @LAST_SEASON')[['season', 'team_id', 'made_playoffs']]
           .merge(rounds, on=['season', 'team_id'], how='left', validate='one_to_one'))

mismatches = history[history['made_playoffs'] != history['furthest_playoff_round'].notna()]
print('team-seasons, 2011-2026:', len(history))
print('made_playoffs disagrees with the playoff games:', len(mismatches))
assert len(mismatches) == 0 and history['made_playoffs'].notna().all()

history['furthest_playoff_round'] = history['furthest_playoff_round'].fillna(0).astype(int)
history['furthest_playoff_round'].value_counts().sort_index().rename('team-seasons').to_frame().T

team-seasons, 2011-2026: 480
made_playoffs disagrees with the playoff games: 0


furthest_playoff_round,0,1,2,3,4,5
team-seasons,224,128,64,32,16,16


**What we found:** The two sources agree for all 480 team-seasons (30 teams × 16 seasons, 2011-2026). Each season has 14 teams with 0, 8 with 1, 4 with 2, 2 with 3, one runner-up with 4, and one champion with 5.

### 4C. 5-year playoff trends

For each team and season we look at the **five previous seasons**: how many of them the team made the playoffs, and its average `furthest_playoff_round`, with missed seasons counted as 0 and the 0-5 scale above. `shift(1)` leaves out the current season, so the trend only describes past seasons. `team_id` stays the same when a franchise is renamed (for example, the Charlotte Bobcats became the Hornets in 2014-15), so each franchise's history is continuous.

In [20]:
history = history.sort_values(['team_id', 'season']).reset_index(drop=True)

# Rolling windows assume every team has every season, with no gaps.
assert (history.groupby('team_id')['season'].agg(lambda s: s.diff().dropna().eq(1).all())).all()

by_team = history.groupby('team_id')
history['playoff_appearances_last_5'] = by_team['made_playoffs'].transform(
    lambda s: s.astype(int).shift(1).rolling(5, min_periods=5).sum())
history['avg_playoff_round_last_5'] = by_team['furthest_playoff_round'].transform(
    lambda s: s.shift(1).rolling(5, min_periods=5).mean())

trends = history.query('@FIRST_SEASON <= season <= @LAST_SEASON').copy()
print('team-seasons with trends:', len(trends))
print('missing trends:', trends[['playoff_appearances_last_5', 'avg_playoff_round_last_5']].isna().sum().to_dict())

# Example: Golden State
gsw = teams_raw.query("nba_abbrev == 'GSW'")['team_id'].iloc[0]
trends[trends['team_id'] == gsw][['season', 'made_playoffs', 'furthest_playoff_round',
                                  'playoff_appearances_last_5', 'avg_playoff_round_last_5']]

team-seasons with trends: 330
missing trends: {'playoff_appearances_last_5': 0, 'avg_playoff_round_last_5': 0}


,season,made_playoffs,furthest_playoff_round,playoff_appearances_last_5,avg_playoff_round_last_5
117,2016,True,4,3.0,1.6
118,2017,True,5,4.0,2.4
119,2018,True,5,5.0,3.4
120,2019,True,4,5.0,4.0
121,2020,False,0,5.0,4.6
122,2021,False,0,4.0,3.6
123,2022,True,5,3.0,2.8
124,2023,True,2,3.0,2.8
125,2024,False,0,3.0,2.2
126,2025,True,2,2.0,1.4


**What we found:** Because 2011-2015 were loaded, **all 330 team-seasons in our window have complete 5-year trends**, with no missing values. Golden State shows the columns working. Its 2015-2019 run of five straight Finals (three titles) pushes `avg_playoff_round_last_5` to 4.6 for 2020. The average then falls as those Finals seasons leave the five-year window and are replaced by the missed playoffs of 2020 and 2021: 3.6 in 2021, 2.8 in 2022 and 2023, and 1.4 by 2025. That is the slow-moving "recent history" signal these columns are meant to capture.

### 4D. Merge the tables

We merge the team information, the regular-season records, and the playoff features on `season` + `team_id`, with `validate='one_to_one'` so a duplicate key would raise an error. We also add `win_pct`.

In [21]:
teams = (team_seasons[['season', 'team_id', 'nba_abbrev', 'full_name', 'conference']]
         .merge(team_stats[['season', 'team_id', 'wins', 'losses', 'g']],
                on=['season', 'team_id'], how='left', validate='one_to_one')
         .merge(trends.drop(columns='made_playoffs'), on=['season', 'team_id'], how='left', validate='one_to_one')
         .merge(team_stats[['season', 'team_id', 'made_playoffs']], on=['season', 'team_id'], how='left', validate='one_to_one'))

teams['win_pct'] = teams['wins'] / teams['g']
print('rows:', len(teams), '| missing values:', int(teams.isna().sum().sum()))

(teams.query('season == 2024 and conference == "East"')
      .sort_values('win_pct', ascending=False)
      [['nba_abbrev', 'wins', 'losses', 'win_pct', 'made_playoffs', 'furthest_playoff_round']])

rows: 330 | missing values: 0


,nba_abbrev,wins,losses,win_pct,made_playoffs,furthest_playoff_round
241,BOS,64,18,0.780488,True,5
255,NYK,50,32,0.609756,True,2
252,MIL,49,33,0.597561,True,1
242,CLE,48,34,0.585366,True,2
258,PHI,47,35,0.573171,True,1
256,ORL,47,35,0.573171,True,1
257,IND,47,35,0.573171,True,3
251,MIA,46,36,0.560976,True,1
244,CHI,39,43,0.475610,False,0
240,ATL,36,46,0.439024,False,0


**What we found:** All 330 team-seasons merge with no missing values. The 2024 East shows why the standing comes from notebook 02 and isn't computed here: ORL, IND, and PHI all went 47-35, so win % alone can't order them, yet their official seeds were 5, 6, and 7, and PHI had to go through the play-in.

### 4E. Keep the feature columns

We drop `cup_group`, `division`, and `is_nba` (not features). We also drop `losses` and `g`, which are already captured by `wins` and `win_pct`. `nba_abbrev` is renamed to `team` to match the join key in the other tables.

In [22]:
team_features = (teams.rename(columns={'nba_abbrev': 'team'})
                 [['season', 'team', 'team_id', 'full_name', 'conference',
                   'wins', 'win_pct',
                   'made_playoffs', 'furthest_playoff_round',
                   'playoff_appearances_last_5', 'avg_playoff_round_last_5']]
                 .astype({'playoff_appearances_last_5': int})
                 .sort_values(['season', 'team'])
                 .reset_index(drop=True))
team_features.head(10)

,season,team,team_id,full_name,conference,wins,win_pct,made_playoffs,furthest_playoff_round,playoff_appearances_last_5,avg_playoff_round_last_5
0,2016,ATL,1610612737,Atlanta Hawks,East,48,0.585366,True,2,5,1.6
1,2016,BKN,1610612751,Brooklyn Nets,East,21,0.256098,False,0,3,0.8
2,2016,BOS,1610612738,Boston Celtics,East,48,0.585366,True,1,4,1.4
3,2016,CHA,1610612766,Charlotte Hornets,East,48,0.585366,True,1,1,0.2
4,2016,CHI,1610612741,Chicago Bulls,East,42,0.512195,False,0,5,1.8
5,2016,CLE,1610612739,Cleveland Cavaliers,East,57,0.695122,True,5,1,0.8
6,2016,DAL,1610612742,Dallas Mavericks,West,42,0.512195,True,1,4,1.6
7,2016,DEN,1610612743,Denver Nuggets,West,33,0.402439,False,0,3,0.6
8,2016,DET,1610612765,Detroit Pistons,East,44,0.536585,True,1,0,0.0
9,2016,GSW,1610612744,Golden State Warriors,West,73,0.890244,True,4,3,1.6


## 5. Validate the final table

In [23]:
tf = team_features
checks = {
    'one row per season + team': not tf.duplicated(['season', 'team']).any(),
    '30 teams in every season': tf.groupby('season')['team'].nunique().eq(30).all(),
    f'seasons are {FIRST_SEASON}-{LAST_SEASON}': tf['season'].between(FIRST_SEASON, LAST_SEASON).all() and tf['season'].nunique() == 11,
    'no missing values': tf.notna().all().all(),
    'playoff round is 0-5': tf['furthest_playoff_round'].between(0, 5).all(),
    '16 playoff teams per season': tf.groupby('season')['made_playoffs'].sum().eq(16).all(),
    'one champion per season': tf.groupby('season')['furthest_playoff_round'].apply(lambda r: (r == 5).sum()).eq(1).all(),
    'round 0 exactly when the team missed the playoffs': (tf['made_playoffs'] == (tf['furthest_playoff_round'] > 0)).all(),
    'appearances in last 5 is 0-5': tf['playoff_appearances_last_5'].between(0, 5).all(),
    'average round in last 5 is 0-5': tf['avg_playoff_round_last_5'].between(0, 5).all(),
}
for name, ok in checks.items():
    print('PASS' if ok else 'FAIL', '-', name)
assert all(checks.values())

PASS - one row per season + team
PASS - 30 teams in every season
PASS - seasons are 2016-2026
PASS - no missing values
PASS - playoff round is 0-5
PASS - 16 playoff teams per season
PASS - one champion per season
PASS - round 0 exactly when the team missed the playoffs
PASS - appearances in last 5 is 0-5
PASS - average round in last 5 is 0-5


In [24]:
tf.drop(columns=['season']).describe().round(2)

,wins,win_pct,furthest_playoff_round,playoff_appearances_last_5,avg_playoff_round_last_5
count,330.00,330.00,330.00,330.00,330.00
mean,40.03,0.50,1.03,2.67,1.03
std,12.19,0.15,1.31,1.60,0.87
min,10.00,0.12,0.00,0.00,0.00
25%,31.25,0.39,0.00,1.00,0.40
50%,42.00,0.51,1.00,3.00,0.80
75%,49.00,0.61,2.00,4.00,1.60
max,73.00,0.89,5.00,5.00,4.60


**What we found:** Every check passes: 330 rows (30 teams × 11 seasons), one row per `season` + `team`, no missing values, 16 playoff teams and one champion per season, and every column within its expected range.

## 6. Save

In [25]:
out = PROCESSED / 'llimllib_team_playoff_features.csv'
team_features.to_csv(out, index=False)
print('saved', team_features.shape, 'to', out)

saved (330, 11) to ..\..\data-files\02-processed\llimllib_team_playoff_features.csv


**Output**

- `data-files/02-processed/llimllib_team_playoff_features.csv`: **one row per team + season** (330 rows, 2015-16 to 2025-26). Keys: `season` (end year) + `team` (NBA abbreviation). Notebook `999` merges it with the ESPN standings from `02` into `team_performance_combined.csv`, which is the table to join onto the player tables.